# 🏜️ Duality AI Offroad Semantic Segmentation - Hackathon

**DINOv2 + Enhanced Segmentation Head**

Steps:
1. Mount Google Drive (upload your dataset there first)
2. Install dependencies
3. Train the model
4. Test on unseen images
5. Download results

⚠️ **Make sure GPU runtime is enabled**: Runtime → Change runtime type → T4 GPU

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q opencv-contrib-python tqdm

## 📁 Setup Paths

**IMPORTANT**: Upload your dataset ZIPs to Google Drive first, then update the paths below.

Expected structure after extraction:
```
/content/data/
  train/
    Color_Images/
    Segmentation/
  val/
    Color_Images/
    Segmentation/
  test/
    Color_Images/
    Segmentation/
```

In [ ]:
import os
import zipfile

# ========================================
# UPDATE THESE PATHS TO MATCH YOUR DRIVE
# ========================================
DRIVE_DATASET_ZIP = '/content/drive/MyDrive/MIT_Hackathon/Offroad_Segmentation_Training_Dataset.zip'
DRIVE_TEST_ZIP = '/content/drive/MyDrive/MIT_Hackathon/Offroad_Segmentation_testImages.zip'

# Working directory on Colab (fast SSD)
WORK_DIR = '/content/hackathon'
os.makedirs(WORK_DIR, exist_ok=True)

# Extract dataset if not already done
data_dir = os.path.join(WORK_DIR, 'Offroad_Segmentation_Training_Dataset')
if not os.path.isdir(data_dir):
    print('Extracting training dataset...')
    with zipfile.ZipFile(DRIVE_DATASET_ZIP, 'r') as z:
        z.extractall(WORK_DIR)
    print('Done!')
else:
    print('Training dataset already extracted.')

# Extract test images if not already done
test_dir = os.path.join(WORK_DIR, 'Offroad_Segmentation_testImages')
if not os.path.isdir(test_dir):
    print('Extracting test images...')
    with zipfile.ZipFile(DRIVE_TEST_ZIP, 'r') as z:
        z.extractall(WORK_DIR)
    print('Done!')
else:
    print('Test images already extracted.')

TRAIN_DIR = os.path.join(data_dir, 'train')
VAL_DIR = os.path.join(data_dir, 'val')
TEST_DIR = test_dir

print(f'\nTrain images: {len(os.listdir(os.path.join(TRAIN_DIR, "Color_Images")))}')
print(f'Val images: {len(os.listdir(os.path.join(VAL_DIR, "Color_Images")))}')
print(f'Test images: {len(os.listdir(os.path.join(TEST_DIR, "Color_Images")))}')

## 🧠 Model Configuration & Training Code

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from PIL import Image
import cv2
import os
import random
import json
import time
from tqdm.auto import tqdm

# ============================================================================
# Configuration
# ============================================================================
VALUE_MAP = {
    0: 0,        # Background
    100: 1,      # Trees
    200: 2,      # Lush Bushes
    300: 3,      # Dry Grass
    500: 4,      # Dry Bushes
    550: 5,      # Ground Clutter
    600: 6,      # Flowers
    700: 7,      # Logs
    800: 8,      # Rocks
    7100: 9,     # Landscape
    10000: 10    # Sky
}

CLASS_NAMES = [
    'Background', 'Trees', 'Lush Bushes', 'Dry Grass', 'Dry Bushes',
    'Ground Clutter', 'Flowers', 'Logs', 'Rocks', 'Landscape', 'Sky'
]

N_CLASSES = len(VALUE_MAP)  # 11

COLOR_PALETTE = np.array([
    [0, 0, 0], [34, 139, 34], [0, 255, 0], [210, 180, 140], [139, 90, 43],
    [128, 128, 0], [255, 105, 180], [139, 69, 19], [128, 128, 128],
    [160, 82, 45], [135, 206, 235]
], dtype=np.uint8)

print(f'Classes: {N_CLASSES}')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {i}: {name}')

In [ ]:
# ============================================================================
# Mask Conversion & Augmentation
# ============================================================================

def convert_mask(mask):
    arr = np.array(mask)
    new_arr = np.zeros_like(arr, dtype=np.uint8)
    for raw_value, new_value in VALUE_MAP.items():
        new_arr[arr == raw_value] = new_value
    return Image.fromarray(new_arr)

def mask_to_color(mask):
    h, w = mask.shape
    color_mask = np.zeros((h, w, 3), dtype=np.uint8)
    for class_id in range(N_CLASSES):
        color_mask[mask == class_id] = COLOR_PALETTE[class_id]
    return color_mask


class PairedTransform:
    def __init__(self, img_size, augment=True):
        self.img_size = img_size
        self.augment = augment
        self.img_normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        )

    def __call__(self, image, mask):
        image = TF.resize(image, self.img_size, interpolation=transforms.InterpolationMode.BILINEAR)
        mask = TF.resize(mask, self.img_size, interpolation=transforms.InterpolationMode.NEAREST)

        if self.augment:
            if random.random() > 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask)
            if random.random() > 0.8:
                image = TF.vflip(image)
                mask = TF.vflip(mask)
            if random.random() > 0.5:
                angle = random.uniform(-15, 15)
                image = TF.rotate(image, angle, fill=0)
                mask = TF.rotate(mask, angle, fill=0)
            if random.random() > 0.3:
                image = transforms.ColorJitter(
                    brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1
                )(image)
            if random.random() > 0.7:
                image = TF.gaussian_blur(image, kernel_size=5, sigma=(0.1, 2.0))

        image = TF.to_tensor(image)
        image = self.img_normalize(image)
        mask = torch.from_numpy(np.array(mask)).long()
        return image, mask


class MaskDataset(Dataset):
    def __init__(self, data_dir, paired_transform=None):
        self.image_dir = os.path.join(data_dir, 'Color_Images')
        self.masks_dir = os.path.join(data_dir, 'Segmentation')
        self.paired_transform = paired_transform
        self.data_ids = sorted(os.listdir(self.image_dir))

    def __len__(self):
        return len(self.data_ids)

    def __getitem__(self, idx):
        data_id = self.data_ids[idx]
        img_path = os.path.join(self.image_dir, data_id)
        mask_path = os.path.join(self.masks_dir, data_id)
        image = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path)
        mask = convert_mask(mask)
        if self.paired_transform:
            image, mask = self.paired_transform(image, mask)
        return image, mask

print('Dataset classes defined.')

In [ ]:
# ============================================================================
# Model Architectures
# ============================================================================

class SegmentationHead(nn.Module):
    def __init__(self, in_channels, out_channels, tokenW, tokenH):
        super().__init__()
        self.H, self.W = tokenH, tokenW
        self.decoder = nn.Sequential(
            nn.Conv2d(in_channels, 256, kernel_size=1),
            nn.BatchNorm2d(256), nn.GELU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.GELU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1, groups=256),
            nn.BatchNorm2d(256), nn.GELU(),
            nn.Conv2d(256, 128, kernel_size=1),
            nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1, groups=128),
            nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 128, kernel_size=1),
            nn.BatchNorm2d(128), nn.GELU(),
        )
        self.classifier = nn.Sequential(
            nn.Dropout2d(0.1),
            nn.Conv2d(128, out_channels, 1)
        )

    def forward(self, x):
        B, N, C = x.shape
        x = x.reshape(B, self.H, self.W, C).permute(0, 3, 1, 2)
        x = self.decoder(x)
        return self.classifier(x)

print('Model architecture defined.')

In [ ]:
# ============================================================================
# Metrics
# ============================================================================

def compute_iou(pred, target, num_classes=N_CLASSES):
    pred = torch.argmax(pred, dim=1)
    pred, target = pred.view(-1), target.view(-1)
    iou_per_class = []
    for class_id in range(num_classes):
        pred_inds = pred == class_id
        target_inds = target == class_id
        intersection = (pred_inds & target_inds).sum().float()
        union = (pred_inds | target_inds).sum().float()
        if union == 0:
            iou_per_class.append(float('nan'))
        else:
            iou_per_class.append((intersection / union).cpu().numpy())
    return np.nanmean(iou_per_class), iou_per_class


def compute_dice(pred, target, num_classes=N_CLASSES, smooth=1e-6):
    pred = torch.argmax(pred, dim=1)
    pred, target = pred.view(-1), target.view(-1)
    dice_per_class = []
    for class_id in range(num_classes):
        pred_inds = pred == class_id
        target_inds = target == class_id
        intersection = (pred_inds & target_inds).sum().float()
        dice_score = (2. * intersection + smooth) / (pred_inds.sum().float() + target_inds.sum().float() + smooth)
        dice_per_class.append(dice_score.cpu().numpy())
    return np.mean(dice_per_class), dice_per_class


def compute_pixel_accuracy(pred, target):
    pred_classes = torch.argmax(pred, dim=1)
    return (pred_classes == target).float().mean().cpu().numpy()


def evaluate_metrics(model, backbone, data_loader, device, num_classes=N_CLASSES):
    iou_scores, dice_scores, pixel_accuracies, all_class_iou = [], [], [], []
    model.eval()
    with torch.no_grad():
        for imgs, labels in data_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            output = backbone.forward_features(imgs)['x_norm_patchtokens']
            logits = model(output)
            outputs = F.interpolate(logits, size=imgs.shape[2:], mode='bilinear', align_corners=False)
            if labels.dim() == 4:
                labels = labels.squeeze(dim=1)
            labels = labels.long()
            iou, class_iou = compute_iou(outputs, labels, num_classes=num_classes)
            dice, _ = compute_dice(outputs, labels, num_classes=num_classes)
            pixel_acc = compute_pixel_accuracy(outputs, labels)
            iou_scores.append(iou)
            dice_scores.append(dice)
            pixel_accuracies.append(pixel_acc)
            all_class_iou.append(class_iou)
    model.train()
    avg_class_iou = np.nanmean(all_class_iou, axis=0)
    return np.nanmean(iou_scores), np.mean(dice_scores), np.mean(pixel_accuracies), avg_class_iou


def compute_class_weights(data_dir, num_classes=N_CLASSES, max_samples=300):
    print('Computing class weights...')
    masks_dir = os.path.join(data_dir, 'Segmentation')
    mask_files = sorted(os.listdir(masks_dir))
    if len(mask_files) > max_samples:
        mask_files = random.sample(mask_files, max_samples)
    pixel_counts = np.zeros(num_classes, dtype=np.float64)
    for mf in tqdm(mask_files, desc='Counting pixels'):
        mask = Image.open(os.path.join(masks_dir, mf))
        mask_arr = np.array(convert_mask(mask))
        for c in range(num_classes):
            pixel_counts[c] += (mask_arr == c).sum()
    total = pixel_counts.sum()
    weights = total / (num_classes * pixel_counts + 1e-6)
    weights = np.clip(weights, 0.1, 10.0)
    weights = weights / weights.mean()
    for name, w in zip(CLASS_NAMES, weights):
        print(f'  {name:<20}: {w:.3f}')
    return torch.FloatTensor(weights)

print('Metrics defined.')

## 🚀 Training

**Recommended settings for T4 GPU (15GB VRAM):**

| Setting | Conservative | Balanced | Aggressive |
|---------|-------------|----------|------------|
| Backbone | small | base | large |
| Batch size | 8 | 4 | 2 |
| Image scale | 0.5 | 0.5 | 0.75 |
| Epochs | 30 | 40 | 50 |
| Expected time | ~20 min | ~45 min | ~2 hours |
| Expected IoU | ~0.35 | ~0.45 | ~0.50+ |

In [ ]:
# ============================================================================
# TRAINING CONFIGURATION - MODIFY THESE
# ============================================================================

# Backbone: 'small' (fastest), 'base' (balanced), 'large' (best quality)
BACKBONE_SIZE = 'base'

# Training hyperparameters
BATCH_SIZE = 4
EPOCHS = 40
LEARNING_RATE = 1e-3
ACCUM_STEPS = 4  # Effective batch = BATCH_SIZE * ACCUM_STEPS
IMG_SCALE = 0.5  # Image resize factor

# Features
USE_AUGMENTATION = True
USE_CLASS_WEIGHTS = True
USE_AMP = True  # Mixed precision (2x faster on GPU)

# Output
OUTPUT_DIR = os.path.join(WORK_DIR, 'runs', f'{BACKBONE_SIZE}_enhanced_ep{EPOCHS}')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Backbone: DINOv2 {BACKBONE_SIZE}')
print(f'Epochs: {EPOCHS}')
print(f'Batch size: {BATCH_SIZE} (effective: {BATCH_SIZE * ACCUM_STEPS})')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
# ============================================================================
# Setup & Train
# ============================================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Image dimensions (divisible by 14 for ViT)
w = int(((960 * IMG_SCALE) // 14) * 14)
h = int(((540 * IMG_SCALE) // 14) * 14)
print(f'Image size: {w}x{h}')

# Transforms
train_transform = PairedTransform((h, w), augment=USE_AUGMENTATION)
val_transform = PairedTransform((h, w), augment=False)

# Datasets
trainset = MaskDataset(TRAIN_DIR, paired_transform=train_transform)
valset = MaskDataset(VAL_DIR, paired_transform=val_transform)

train_loader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(valset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

print(f'Train: {len(trainset)}, Val: {len(valset)}')

# Load DINOv2 backbone
backbone_archs = {'small': 'vits14', 'base': 'vitb14_reg', 'large': 'vitl14_reg'}
backbone_name = f'dinov2_{backbone_archs[BACKBONE_SIZE]}'
print(f'\nLoading {backbone_name}...')
backbone_model = torch.hub.load('facebookresearch/dinov2', backbone_name)
backbone_model.eval()
backbone_model.to(device)

# Get embedding dimension
with torch.no_grad():
    sample = trainset[0][0].unsqueeze(0).to(device)
    tokens = backbone_model.forward_features(sample)['x_norm_patchtokens']
    n_embedding = tokens.shape[2]
print(f'Embedding dim: {n_embedding}, Patch tokens: {tokens.shape[1]}')

# Segmentation head
classifier = SegmentationHead(
    in_channels=n_embedding, out_channels=N_CLASSES,
    tokenW=w//14, tokenH=h//14
).to(device)

# Class weights
if USE_CLASS_WEIGHTS:
    class_weights = compute_class_weights(TRAIN_DIR).to(device)
    loss_fct = nn.CrossEntropyLoss(weight=class_weights)
else:
    loss_fct = nn.CrossEntropyLoss()

# Optimizer & Scheduler
optimizer = optim.AdamW(classifier.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda') if USE_AMP else None

print('\n✅ Setup complete! Ready to train.')

In [ ]:
# ============================================================================
# TRAINING LOOP
# ============================================================================

history = {
    'train_loss': [], 'val_loss': [],
    'train_iou': [], 'val_iou': [],
    'train_dice': [], 'val_dice': [],
    'train_pixel_acc': [], 'val_pixel_acc': [],
    'lr': []
}

best_val_iou = 0
best_epoch = 0
total_start = time.time()

print(f'Starting training for {EPOCHS} epochs...')
print('=' * 80)

for epoch in range(EPOCHS):
    epoch_start = time.time()

    # --- Train ---
    classifier.train()
    train_losses = []
    optimizer.zero_grad()

    for step, (imgs, labels) in enumerate(tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]', leave=False)):
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.no_grad():
            features = backbone_model.forward_features(imgs)['x_norm_patchtokens']

        if USE_AMP:
            with torch.amp.autocast('cuda'):
                logits = classifier(features)
                outputs = F.interpolate(logits, size=imgs.shape[2:], mode='bilinear', align_corners=False)
                loss = loss_fct(outputs, labels) / ACCUM_STEPS
            scaler.scale(loss).backward()
        else:
            logits = classifier(features)
            outputs = F.interpolate(logits, size=imgs.shape[2:], mode='bilinear', align_corners=False)
            loss = loss_fct(outputs, labels) / ACCUM_STEPS
            loss.backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            if USE_AMP:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad()

        train_losses.append(loss.item() * ACCUM_STEPS)

    # --- Validate ---
    classifier.eval()
    val_losses = []
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Val]', leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            features = backbone_model.forward_features(imgs)['x_norm_patchtokens']
            logits = classifier(features)
            outputs = F.interpolate(logits, size=imgs.shape[2:], mode='bilinear', align_corners=False)
            loss = loss_fct(outputs, labels)
            val_losses.append(loss.item())

    # --- Metrics (every 5 epochs to save time) ---
    if (epoch + 1) % 5 == 0 or epoch == 0 or epoch == EPOCHS - 1:
        train_iou, train_dice, train_acc, _ = evaluate_metrics(classifier, backbone_model, train_loader, device)
        val_iou, val_dice, val_acc, val_class_iou = evaluate_metrics(classifier, backbone_model, val_loader, device)
    # else just carry forward last values
    elif len(history['train_iou']) > 0:
        train_iou = history['train_iou'][-1]
        train_dice = history['train_dice'][-1]
        train_acc = history['train_pixel_acc'][-1]
        val_iou = history['val_iou'][-1]
        val_dice = history['val_dice'][-1]
        val_acc = history['val_pixel_acc'][-1]
    else:
        train_iou = val_iou = train_dice = val_dice = train_acc = val_acc = 0

    # Store
    epoch_train_loss = np.mean(train_losses)
    epoch_val_loss = np.mean(val_losses)
    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(float(epoch_train_loss))
    history['val_loss'].append(float(epoch_val_loss))
    history['train_iou'].append(float(train_iou))
    history['val_iou'].append(float(val_iou))
    history['train_dice'].append(float(train_dice))
    history['val_dice'].append(float(val_dice))
    history['train_pixel_acc'].append(float(train_acc))
    history['val_pixel_acc'].append(float(val_acc))
    history['lr'].append(current_lr)

    scheduler.step()
    epoch_time = time.time() - epoch_start

    print(f'Epoch {epoch+1}/{EPOCHS} ({epoch_time:.0f}s) | '
          f'Loss: {epoch_train_loss:.4f}/{epoch_val_loss:.4f} | '
          f'IoU: {train_iou:.4f}/{val_iou:.4f} | '
          f'LR: {current_lr:.6f}')

    # Save best
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        best_epoch = epoch + 1
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': classifier.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_iou': val_iou,
        }, os.path.join(OUTPUT_DIR, 'best_model.pth'))
        print(f'  ★ New best! IoU: {val_iou:.4f}')

total_time = time.time() - total_start
print(f'\n{"="*80}')
print(f'Training complete! {total_time/60:.1f} minutes')
print(f'Best Val IoU: {best_val_iou:.4f} (Epoch {best_epoch})')
print(f'{"="*80}')

In [ ]:
# ============================================================================
# FINAL EVALUATION & PLOTS
# ============================================================================

# Load best model
best_ckpt = torch.load(os.path.join(OUTPUT_DIR, 'best_model.pth'), map_location=device)
classifier.load_state_dict(best_ckpt['model_state_dict'])
_, _, _, final_class_iou = evaluate_metrics(classifier, backbone_model, val_loader, device)
history['final_class_iou'] = [float(x) for x in final_class_iou]

print('Per-Class IoU (Best Model):')
for name, iou in zip(CLASS_NAMES, final_class_iou):
    s = f'{iou:.4f}' if not np.isnan(iou) else 'N/A'
    print(f'  {name:<20}: {s}')
print(f'\nMean IoU: {np.nanmean(final_class_iou):.4f}')

# Save plots
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes[0,0].plot(history['train_loss'], label='Train'); axes[0,0].plot(history['val_loss'], label='Val')
axes[0,0].set_title('Loss'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)
axes[0,1].plot(history['train_iou'], label='Train'); axes[0,1].plot(history['val_iou'], label='Val')
axes[0,1].set_title('IoU'); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)
axes[1,0].plot(history['train_dice'], label='Train'); axes[1,0].plot(history['val_dice'], label='Val')
axes[1,0].set_title('Dice'); axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)
axes[1,1].plot(history['lr'])
axes[1,1].set_title('Learning Rate'); axes[1,1].grid(True, alpha=0.3)
plt.suptitle(f'Training Metrics (Best IoU: {best_val_iou:.4f} @ Epoch {best_epoch})', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150)
plt.show()

# Per-class IoU bar chart
fig, ax = plt.subplots(figsize=(12, 6))
valid_iou = [x if not np.isnan(x) else 0 for x in final_class_iou]
colors = plt.cm.Set3(np.linspace(0, 1, N_CLASSES))
bars = ax.bar(range(N_CLASSES), valid_iou, color=colors, edgecolor='black')
ax.set_xticks(range(N_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_ylabel('IoU')
ax.set_title(f'Per-Class IoU (Mean: {np.nanmean(final_class_iou):.4f})')
ax.set_ylim(0, 1)
ax.axhline(y=np.nanmean(final_class_iou), color='red', linestyle='--', label='Mean')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, valid_iou):
    if val > 0:
        ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.01, f'{val:.3f}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'per_class_iou.png'), dpi=150)
plt.show()

# Save weights separately
torch.save(classifier.state_dict(), os.path.join(OUTPUT_DIR, 'segmentation_head.pth'))

# Save history JSON
with open(os.path.join(OUTPUT_DIR, 'training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)

print(f'\nAll results saved to {OUTPUT_DIR}')

## 🧪 Test on Unseen Images

In [ ]:
# ============================================================================
# TEST ON UNSEEN IMAGES
# ============================================================================

TEST_OUTPUT_DIR = os.path.join(WORK_DIR, 'test_predictions')
os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(TEST_OUTPUT_DIR, 'masks'), exist_ok=True)
os.makedirs(os.path.join(TEST_OUTPUT_DIR, 'masks_color'), exist_ok=True)
os.makedirs(os.path.join(TEST_OUTPUT_DIR, 'comparisons'), exist_ok=True)

# Load best model
best_ckpt = torch.load(os.path.join(OUTPUT_DIR, 'best_model.pth'), map_location=device)
classifier.load_state_dict(best_ckpt['model_state_dict'])
classifier.eval()

# Test dataset
class TestDataset(Dataset):
    def __init__(self, data_dir, img_size):
        self.image_dir = os.path.join(data_dir, 'Color_Images')
        self.masks_dir = os.path.join(data_dir, 'Segmentation')
        self.data_ids = sorted(os.listdir(self.image_dir))
        self.img_size = img_size
        self.has_masks = os.path.isdir(self.masks_dir)
        self.normalize = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

    def __len__(self):
        return len(self.data_ids)

    def __getitem__(self, idx):
        data_id = self.data_ids[idx]
        image = Image.open(os.path.join(self.image_dir, data_id)).convert('RGB')
        image = TF.resize(image, self.img_size, interpolation=transforms.InterpolationMode.BILINEAR)
        image = TF.to_tensor(image)
        image = self.normalize(image)
        mask = None
        if self.has_masks:
            mask_path = os.path.join(self.masks_dir, data_id)
            if os.path.exists(mask_path):
                mask = Image.open(mask_path)
                mask = convert_mask(mask)
                mask = TF.resize(mask, self.img_size, interpolation=transforms.InterpolationMode.NEAREST)
                mask = torch.from_numpy(np.array(mask)).long()
        return image, mask, data_id

test_dataset = TestDataset(TEST_DIR, (h, w))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Test images: {len(test_dataset)}')
print(f'Has ground truth: {test_dataset.has_masks}')

# Run inference
all_iou = []
all_class_iou = []
confusion_matrix = np.zeros((N_CLASSES, N_CLASSES), dtype=np.int64)
inference_times = []
vis_count = 0

with torch.no_grad():
    for imgs, masks, data_ids in tqdm(test_loader, desc='Testing'):
        imgs = imgs.to(device)
        t0 = time.time()
        features = backbone_model.forward_features(imgs)['x_norm_patchtokens']
        logits = classifier(features)
        outputs = F.interpolate(logits, size=imgs.shape[2:], mode='bilinear', align_corners=False)
        predicted = torch.argmax(outputs, dim=1)
        inference_times.append((time.time() - t0) / imgs.shape[0])

        # Metrics if GT available
        if test_dataset.has_masks and masks[0] is not None:
            gt = torch.stack([m for m in masks]).to(device)
            iou, class_iou = compute_iou(outputs, gt)
            all_iou.append(iou)
            all_class_iou.append(class_iou)
            # Confusion matrix
            p = predicted.view(-1).cpu().numpy()
            t = gt.view(-1).cpu().numpy()
            for ti, pi in zip(t, p):
                if ti < N_CLASSES and pi < N_CLASSES:
                    confusion_matrix[ti][pi] += 1

        # Save all predictions
        for i in range(imgs.shape[0]):
            data_id = data_ids[i]
            base = os.path.splitext(data_id)[0]
            pred_mask = predicted[i].cpu().numpy().astype(np.uint8)
            Image.fromarray(pred_mask).save(os.path.join(TEST_OUTPUT_DIR, 'masks', f'{base}_pred.png'))
            pred_color = mask_to_color(pred_mask)
            cv2.imwrite(os.path.join(TEST_OUTPUT_DIR, 'masks_color', f'{base}_pred_color.png'),
                        cv2.cvtColor(pred_color, cv2.COLOR_RGB2BGR))
            if vis_count < 30:
                # Denormalize for vis
                img_np = imgs[i].cpu().numpy()
                img_np = np.moveaxis(img_np, 0, -1) * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])
                img_np = np.clip(img_np, 0, 1)
                fig, axes = plt.subplots(1, 2, figsize=(12, 5))
                axes[0].imshow(img_np); axes[0].set_title('Input'); axes[0].axis('off')
                axes[1].imshow(pred_color); axes[1].set_title('Prediction'); axes[1].axis('off')
                plt.suptitle(data_id)
                plt.tight_layout()
                plt.savefig(os.path.join(TEST_OUTPUT_DIR, 'comparisons', f'test_{vis_count:04d}.png'), dpi=100)
                plt.close()
                vis_count += 1

print(f'\nAvg inference time: {np.mean(inference_times)*1000:.1f} ms/image')

if len(all_iou) > 0:
    mean_iou = np.nanmean(all_iou)
    avg_class_iou = np.nanmean(all_class_iou, axis=0)
    print(f'Test Mean IoU: {mean_iou:.4f}')
    for name, iou in zip(CLASS_NAMES, avg_class_iou):
        s = f'{iou:.4f}' if not np.isnan(iou) else 'N/A'
        print(f'  {name:<20}: {s}')

print(f'\nSaved {vis_count} test visualizations to {TEST_OUTPUT_DIR}')

## 💾 Download Results

In [ ]:
# Copy results to Google Drive for persistence
import shutil

DRIVE_SAVE_DIR = '/content/drive/MyDrive/MIT_Hackathon/results'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# Copy model weights
shutil.copy2(os.path.join(OUTPUT_DIR, 'best_model.pth'), os.path.join(DRIVE_SAVE_DIR, 'best_model.pth'))
shutil.copy2(os.path.join(OUTPUT_DIR, 'segmentation_head.pth'), os.path.join(DRIVE_SAVE_DIR, 'segmentation_head.pth'))

# Copy training plots
for f in ['training_curves.png', 'per_class_iou.png', 'training_history.json']:
    src = os.path.join(OUTPUT_DIR, f)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DRIVE_SAVE_DIR, f))

# Copy test predictions (zip for size)
shutil.make_archive(os.path.join(DRIVE_SAVE_DIR, 'test_predictions'), 'zip', TEST_OUTPUT_DIR)

print(f'\n✅ All results saved to Google Drive: {DRIVE_SAVE_DIR}')
!ls -la {DRIVE_SAVE_DIR}